# Grad Café pipeline: S3 to SageMaker

This notebook downloads the Grad Café applicant data from S3 with boto3 and saves it next to the
notebook as `applicant_data_SM.json`, ready for cleaning and model preparation.

* **Source:** `s3://grad-cafe-msasso1/applicant_data.json` in `us-west-2`. The bucket name is set once,
  in `DEFAULT_BUCKET` at the top of `s3_fetch.py` (the `GRAD_CAFE_BUCKET` environment variable overrides it).
* **Code:** the boto3 logic lives in `s3_fetch.py`, which must sit in the same folder as this notebook.
* **Credentials:** none in this notebook. `s3_fetch` opens a `boto3.Session()`, and boto3's default
  credential chain finds the notebook instance's IAM execution role. The download cell's report names the
  credential source (on SageMaker, the notebook instance's role), never the keys.

In [1]:
# pylint: disable=invalid-name  # the required file name grad-cafe-pipeline has hyphens
"""Download applicant_data.json from S3 with boto3 and save it as applicant_data_SM.json."""
# pylint: enable=invalid-name
import json
from collections import Counter

from s3_fetch import DEFAULT_KEY, DEFAULT_OUTPUT, default_bucket, fetch_applicant_data

## Settings

`default_bucket()` returns `DEFAULT_BUCKET` from `s3_fetch.py` (`grad-cafe-msasso1`) unless the
`GRAD_CAFE_BUCKET` environment variable names another bucket. If your bucket has another name,
change `DEFAULT_BUCKET` in `s3_fetch.py` and upload that file again; leave this cell as it is.

In [2]:
BUCKET = default_bucket()
print(f"Source: s3://{BUCKET}/{DEFAULT_KEY}")
print(f"Output: {DEFAULT_OUTPUT} (in this notebook's folder)")

Source: s3://grad-cafe-msasso1/applicant_data.json
Output: applicant_data_SM.json (in this notebook's folder)


## Download

`fetch_applicant_data` checks the object with `head_object`, downloads it with `download_file`,
checks that the size matches and that the file is a JSON list, and only then saves it as
`applicant_data_SM.json`.

In [3]:
result = fetch_applicant_data(BUCKET, DEFAULT_KEY, DEFAULT_OUTPUT)
print(result.describe())

Downloaded s3://grad-cafe-msasso1/applicant_data.json
Saved as   /home/ec2-user/SageMaker/applicant_data_SM.json
Size       36,355,821 bytes
Records    30,500
Credentials came from boto3's default chain: iam-role


## A first look at the saved file

Read `applicant_data_SM.json` back from the notebook's disk (not from S3) to confirm the copy is usable.

In [4]:
with open(DEFAULT_OUTPUT, encoding="utf-8") as handle:
    applicants = json.load(handle)

print(f"{len(applicants):,} records in {DEFAULT_OUTPUT}")
print("Fields:", ", ".join(sorted(applicants[0])))
for status, count in Counter(row.get("status") for row in applicants).most_common():
    print(f"  {status}: {count:,}")

30,500 records in applicant_data_SM.json
Fields: comments, date_added, decision_date, decision_date_source, degree, gpa, gre, gre_aw, gre_v, llm-generated-program, llm-generated-university, other_tags, program, program_name, raw, result_id, status, term, university, url, us_or_international
  Rejected: 14,208
  Accepted: 11,371
  Waitlisted: 2,735
  Interview: 2,186
